# 06 - Regularization: Dropout & Batch Normalization

Continues directly from `05_ann_fashion_mnist_pytorch_gpu.ipynb`. Same dataset, same
ANN architecture -- but this time we deliberately **overfit** it first (so you can see
the problem with your own eyes), then fix it with **Dropout** and **BatchNorm**.

**What you'll see:**
1. A model trained with almost no regularization, on purpose, until it overfits
2. Reading the train/test loss gap to diagnose overfitting
3. Adding Dropout -- what it does and why it only activates during training
4. Adding BatchNorm -- what it does and why it needs `.eval()` mode too
5. Comparing all three models on the same held-out test set


In [ ]:
# Same imports as notebook 05
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import os

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
CSV_PATH = 'fmnist_small.csv'

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded real Fashion-MNIST data from '{CSV_PATH}'")
    # Deliberately keep only a SMALL slice of the real data -- overfitting is easiest
    # to demonstrate when there's too little data relative to model capacity, which is
    # exactly the classic real-world setup (small datasets, big networks).
    df = df.sample(n=min(400, len(df)), random_state=42).reset_index(drop=True)
else:
    print(f"'{CSV_PATH}' not found -- generating a small SYNTHETIC placeholder dataset.")
    rng = np.random.RandomState(42)
    n_samples = 400   # deliberately SMALL -- see comment above
    labels = rng.randint(0, 10, n_samples)
    class_patterns = rng.randint(50, 200, size=(10, 784))
    # Heavier per-sample noise (comparable in scale to the signal) makes the underlying
    # pattern genuinely hard to learn from few examples -- a harder, more realistic task
    # than near-perfectly-separable clusters, so overfitting is easy to induce.
    pixels = class_patterns[labels] + rng.randint(-90, 90, size=(n_samples, 784))
    pixels = np.clip(pixels, 0, 255)
    df = pd.DataFrame(pixels, columns=[f'pixel{i}' for i in range(784)])
    df.insert(0, 'label', labels)

df.fillna(0, inplace=True)
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train, X_test = X_train/255.0, X_test/255.0

# On top of the small dataset, corrupt a slice of TRAINING labels only (test stays
# clean). Combined with the small sample count, this makes the train/test GAP
# reliably visible: perfectly fitting the corrupted labels requires memorization
# that cannot help (and actively hurts) performance on the clean test set.
noise_rng = np.random.RandomState(0)
noisy_idx = noise_rng.choice(len(y_train), size=int(len(y_train)*0.2), replace=False)
y_train = y_train.copy()
y_train[noisy_idx] = noise_rng.randint(0, 10, size=len(noisy_idx))

print("train:", X_train.shape, " test:", X_test.shape)
print(f"(small training set + {len(noisy_idx)} corrupted training labels -- the "
      "classic combination for reliably demonstrating overfitting)")
print("NOTE: real Fashion-MNIST data shows this pattern even more naturally, "
      "thanks to genuine label ambiguity between visually similar classes.")

In [ ]:
# Same CustomDataset pattern as notebook 05
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)
  def __len__(self):
    return len(self.features)
  def __getitem__(self, index):
    return self.features[index], self.labels[index]

train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## Step 1: Deliberately Overfit (no regularization)

A network with plenty of capacity (lots of parameters) relative to a small dataset,
trained for many epochs, with no regularization at all -- textbook overfitting setup.
Watch the train/test loss gap widen as training continues.


In [ ]:
# A deliberately OVER-capacity network (way more parameters than this tiny
# dataset needs) with NO dropout, NO batchnorm -- built to overfit on purpose.
class OverfitNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Linear(256, 256),
        nn.ReLU(),
        nn.Linear(256, 10)
    )
  def forward(self, x):
    return self.model(x)

def train_and_evaluate(model, epochs=40, lr=0.01, label="model"):
    """Reusable training loop (same 5-step rhythm as every other notebook in this
    course) that also records train/test loss per epoch, so we can plot and compare."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    train_losses, test_losses = [], []

    for epoch in range(epochs):
        model.train()   # IMPORTANT: enables Dropout / BatchNorm's training behavior
        total_train_loss = 0
        for batch_features, batch_labels in train_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        train_losses.append(total_train_loss / len(train_loader))

        model.eval()    # IMPORTANT: disables Dropout / freezes BatchNorm running stats
        total_test_loss = 0
        with torch.no_grad():
            for batch_features, batch_labels in test_loader:
                batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
                outputs = model(batch_features)
                total_test_loss += criterion(outputs, batch_labels).item()
        test_losses.append(total_test_loss / len(test_loader))

    print(f"[{label}] final train loss: {train_losses[-1]:.4f}  "
          f"final test loss: {test_losses[-1]:.4f}  "
          f"gap: {test_losses[-1]-train_losses[-1]:+.4f}")
    return train_losses, test_losses

overfit_model = OverfitNN(X_train.shape[1])
overfit_train, overfit_test = train_and_evaluate(overfit_model, epochs=40, label="overfit (no regularization)")

## Step 2: Add Dropout

`nn.Dropout(p)` randomly zeroes out a fraction `p` of neurons on EVERY forward pass
DURING TRAINING ONLY. The network can't rely on any single neuron always being present,
which forces it to learn more redundant, robust features instead of memorizing.


In [ ]:
class DropoutNN(nn.Module):
  def __init__(self, num_features, dropout_p=0.3):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Dropout(dropout_p),      # randomly zero 30% of activations during training
        nn.Linear(256, 256),
        nn.ReLU(),
        nn.Dropout(dropout_p),
        nn.Linear(256, 10)
    )
  def forward(self, x):
    return self.model(x)

dropout_model = DropoutNN(X_train.shape[1], dropout_p=0.3)
dropout_train, dropout_test = train_and_evaluate(dropout_model, epochs=40, label="with Dropout(0.3)")

**A quirk worth knowing:** Dropout's own TRAIN loss can look *higher* than its
TEST loss. This isn't a bug -- Dropout is active during `model.train()` (randomly
zeroing neurons on every forward pass, which adds noise to the training-time loss)
but switched off during `model.eval()` for a clean, full-capacity forward pass. So
"train loss > test loss" for a dropout model is often completely normal, not a sign
of underfitting.


## Step 3: Add Batch Normalization

`nn.BatchNorm1d` normalizes each layer's activations (zero mean, unit variance) using
the statistics of the current mini-batch during training, and a running average during
evaluation. This stabilizes and speeds up training, and has a mild regularizing effect
too (the batch statistics add a small amount of noise).


In [ ]:
class BatchNormNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 256),
        nn.BatchNorm1d(256),        # normalize activations before the next layer
        nn.ReLU(),
        nn.Linear(256, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Linear(256, 10)
    )
  def forward(self, x):
    return self.model(x)

batchnorm_model = BatchNormNN(X_train.shape[1])
bn_train, bn_test = train_and_evaluate(batchnorm_model, epochs=40, label="with BatchNorm")

## Step 4: Compare All Three

Same architecture size, same data, same epochs -- the only difference is regularization.


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for a, (train_l, test_l, title) in zip(ax, [
    (overfit_train, overfit_test, "No regularization\n(overfits)"),
    (dropout_train, dropout_test, "With Dropout"),
    (bn_train, bn_test, "With BatchNorm"),
]):
    a.plot(train_l, label="train")
    a.plot(test_l, label="test")
    a.set_title(title)
    a.set_xlabel("epoch")
    a.legend()

ax[0].set_ylabel("loss")
plt.tight_layout()
plt.show()

print(f"{'model':30s} {'train->test gap':>18s}")
for train_l, test_l, name in [
    (overfit_train, overfit_test, "no regularization"),
    (dropout_train, dropout_test, "dropout"),
    (bn_train, bn_test, "batchnorm"),
]:
    gap = test_l[-1] - train_l[-1]
    print(f"{name:30s} {gap:>+18.4f}")

### What to look for

- The **no-regularization** model's test loss should plateau or start rising while
  train loss keeps falling -- that widening gap IS overfitting, visible directly in the plot.
- **Dropout** and **BatchNorm** should each show a smaller (or non-widening) gap --
  regularization working as intended.
- On real Fashion-MNIST (not the synthetic placeholder), this effect is even clearer
  with more training data and more epochs.

**Common trap:** forgetting `model.eval()` before evaluation. Dropout stays "on"
(randomly zeroing neurons) and BatchNorm keeps using batch statistics instead of its
learned running average -- both make evaluation numbers meaningless if you forget this.
